# Oplit Technical Interview — Workshop & Production Times

**The context.** Oplit helps factories **schedule their production**. A new client (a machining / welding / assembly workshop) wants help planning its operations. We'll start from *their* problem, not from the code.

## Part A — Workshop scheduling (flow shop)

The workshop makes 3 part types (A, B, C) on **2 machines**. Each part goes through Machine 1 **then** Machine 2 (in that order). A machine processes only one part at a time.

| Part | Machine 1 | Machine 2 |
|------|-----------|-----------|
| A    | 2 h       | 3 h       |
| B    | 4 h       | 1 h       |
| C    | 3 h       | 2 h       |

The workshop manager says: *“I just want to get everything out as fast as possible.”*

💬 **To discuss (no mathematical model expected):**
- What are we trying to optimize, in the client's words and then in yours?
- What do we actually **decide** here? Which **constraints** can't be violated?
- What **simple rule** (heuristic) could a human planner apply? Why would it be reasonable?
- More generally: faced with *any* optimization problem, what is your **checklist** before writing a single line of code?

## Part B — Predicting processing times

Problem: everything we just did assumes we **know** the duration of each operation (2 h, 4 h…). In the real world, the same operation takes a different time depending on the part, the machine, the batch size, and even the state of the factory at time t.

The client has no standard times of their own — they have a **raw export of what actually happened** on the workshop floor. So the scheduling problem silently depends on a **prediction** problem: *how long will this operation take?*

⚠️ Catch: the **actual duration is not a column in the file**. The export only has **timestamps** (actual start / end). You have to reconstruct the target.

Here is a sample of the client export. **React to what you see.**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works locally (cloned repo) AND on Google Colab (auto-download from GitHub)
LOCAL_PATH = "../data/operations_raw.csv"
RAW_URL = "https://raw.githubusercontent.com/alexandrerioo/scheduling_sandbox/main/data/operations_raw.csv"
CSV = LOCAL_PATH if os.path.exists(LOCAL_PATH) else RAW_URL

df = pd.read_csv(CSV, dtype=str)
print(f"{len(df):,} rows  x  {df.shape[1]} columns")
df.head(8)

### Column glossary

| Column | Description |
|---|---|
| `of_id` | Manufacturing Order (work order) — **1 OF = 1 operation** |
| `article_ref`, `article_family` | Article reference and family (Machining, Welding, Assembly, Finishing) |
| `diameter_mm` | Part diameter (mm) — **article characteristic** |
| `material` | Material: Aluminium / Steel / Stainless / Titanium — **article characteristic** |
| `color` | Part color — **article characteristic** |
| `quantity` | Operation quantity |
| `machine_id`, `machine_type` | Physical machine and its type (different speed) |
| `planned_start_ts` | **Planned** start of the operation (schedule) |
| `actual_start_ts`, `actual_end_ts` | **Actual** start / end of the operation |
| `status` | done / running / aborted |
| `scrap_qty` | Quantity scrapped during the operation |
| `record_created_ts` | Timestamp the row was written by the MES |

🎯 **The target is not provided.** The actual operation duration is reconstructed as: `actual_end_ts − actual_start_ts`.

📌 Several OFs share the same article → you can **aggregate by article** (the `diameter_mm`, `material`, `color` characteristics are constant per reference).

In [ ]:
# Volume & period
# NB: date formats are not homogeneous across sources -> we parse both
iso = pd.to_datetime(df["planned_start_ts"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
fr  = pd.to_datetime(df["planned_start_ts"], format="%d/%m/%Y %H:%M", errors="coerce")
ps = iso.fillna(fr)
print("Period:", ps.min(), "->", ps.max())
print("Distinct work orders  :", df["of_id"].nunique())
print("Article references     :", df["article_ref"].nunique(), "(surface forms)")
print("Materials / colors     :", df["material"].nunique(), "/", df["color"].nunique())
print("Machines / types       :", df["machine_id"].nunique(), "/", df["machine_type"].nunique())
print("actual_end_ts missing  :", (df["actual_end_ts"].fillna('') == '').sum(), "rows (target to reconstruct)")
print("Duplicated rows        :", df.duplicated().sum())

### 💬 To discuss with your interviewer

No need to code everything: we want your **approach** and your **trade-offs**.

1. **Ingestion & quality.** The real client file is tens of GB and doesn't fit in memory. How do you ingest it? And this data is incomplete/inconsistent. *What do you do first?*
2. **Framing.** The actual duration is not a column: how do you **reconstruct** it, and what do you do when `actual_end_ts` is missing or the duration is ≤ 0? At what **granularity** do we model? Which columns **can't you** use to predict, and why?
3. **Split.** How do you separate train / test? (several months of history, we want to predict the future)
4. **Features & model.** Among the article characteristics (`diameter_mm`, `material`, `color`), which look **genuinely** predictive and how do you check? What other variables do you build? What first model, and what **baseline** to beat (e.g. per-article mean, obtained by aggregation)?
5. **Evaluation.** How do you know the model is good, and how do you explain it to the workshop manager?
6. **Production.** Once deployed, the model feeds the scheduler from Part A. What can happen in production, and what does the client gain?

In [ ]:
# Free workspace
